# Campañas y variedades por estación con datos climáticos

Para cada estación de fenología que tiene archivo de clima (`datos/raw/estaciones`), cuenta:
- **campañas**: estación + cultivo + fecha de siembra (misma definición que `scripts/fases_maiz.py`:
  si la misma fecha de siembra reaparece tras más de 70 días sin reportes, es otra campaña);
- **variedades** distintas (minúsculas, sin tildes, sinónimos unificados);
- **cobertura climática** de cada campaña: % de días con Tmax y Tmin entre la siembra y el último reporte.
  Indica si la campaña sirve para calcular grados-día.

Requiere haber ejecutado antes `relacion_estaciones.ipynb` (genera `relacion_estaciones.csv` y
`clima_estaciones_diario.csv`).

**Salida**: `datos/processed/campanias_estaciones_clima.xlsx`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "scripts"))
from fases_maiz import GAP_DIAS, leer_base, moda, norm_variedad

PROC = RAIZ / "datos" / "processed"

# --- Parámetros
CULTIVO = r"\bmaiz\b|choclo"   # regex sobre cultivo_norm; None = todos los cultivos
COBERTURA_MIN = 90             # % de días con Tmax y Tmin para considerar la campaña "con clima completo"
MAX_DIAS_CAMPANIA = 365        # tope del período de una campaña para medir la cobertura

pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)

## 1. Datos

In [ ]:
rel = pd.read_csv(PROC / "relacion_estaciones.csv", dtype={"codigo": str})
rel = rel[rel["duplicado_de"].isna() & rel["estacion_fenologia"].notna()]
est_clima = rel[["estacion_fenologia", "ddrr", "codigo", "archivo"]].drop_duplicates(
    ["estacion_fenologia", "ddrr"])

clima = pd.read_csv(PROC / "clima_estaciones_diario.csv", dtype={"codigo": str}, parse_dates=["fecha"])
clima["temp_ok"] = clima["tmax"].notna() & clima["tmin"].notna()

fen = leer_base(PROC / "FENOLOGIA_consolidado.csv")     # usa ddrr_norm como ddrr
fen = fen.merge(est_clima, left_on=["estacion_norm", "ddrr"],
                right_on=["estacion_fenologia", "ddrr"], how="inner")
fen = fen[fen["tipo_registro"] == "observacion"].copy()
fen["variedad_norm"] = [norm_variedad(v, e) for v, e in zip(fen["variedad"], fen["estacion_norm"])]
print(f"{len(est_clima)} estaciones con clima | {len(fen):,} observaciones de fenología en ellas")

## 2. Campañas

Una campaña es estación + cultivo + fecha de siembra. Los registros sin fecha de siembra no se cuentan.
Si la misma fecha de siembra vuelve a aparecer después de más de `GAP_DIAS` días sin reportes, es otra campaña
(pasa cuando la fecha se copia de un año a otro).

In [ ]:
def campanias(df):
    d = df[df["fecha_siembra"].notna() & df["cultivo_norm"].notna()].sort_values(
        ["estacion_norm", "cultivo_norm", "fecha_siembra", "semana_inicio"]).copy()
    llave = ["estacion_norm", "ddrr", "cultivo_norm", "fecha_siembra"]
    salto = d.groupby(llave)["semana_inicio"].diff().dt.days
    d["tramo"] = (salto > GAP_DIAS).groupby([d[k] for k in llave]).cumsum()
    c = d.groupby(llave + ["tramo"]).agg(
        codigo=("codigo", "first"),
        variedad=("variedad_norm", moda),
        primera_semana=("semana_inicio", "min"),
        ultima_semana=("semana_inicio", "max"),
        semanas=("semana_inicio", "nunique"),
    ).reset_index().drop(columns="tramo")
    # Período a evaluar: desde la siembra (o desde el primer reporte, si la siembra es muy anterior,
    # como en frutales) hasta el fin de la última semana reportada
    lejos = (c["primera_semana"] - c["fecha_siembra"]).dt.days > 60
    c["perenne_o_antigua"] = lejos
    c["inicio"] = c["fecha_siembra"].where(~lejos, c["primera_semana"])
    c["fin"] = (c["ultima_semana"] + pd.Timedelta(days=6)).clip(
        upper=c["inicio"] + pd.Timedelta(days=MAX_DIAS_CAMPANIA))
    return c


def cobertura(c, clima):
    """% de días del período de cada campaña con Tmax y Tmin en la estación climática."""
    ok = clima[clima["temp_ok"]].groupby("codigo")["fecha"].apply(set)
    rango = clima.groupby("codigo")["fecha"].agg(["min", "max"])
    pct, dentro = [], []
    for r in c.itertuples():
        dias = pd.date_range(r.inicio, r.fin, freq="D")
        if len(dias) == 0:                  # siembra posterior al último reporte
            pct.append(None)
            dentro.append(None)
            continue
        fechas = ok.get(r.codigo, set())
        pct.append(round(100 * sum(f in fechas for f in dias) / len(dias), 1))
        dentro.append(r.inicio >= rango.at[r.codigo, "min"] and r.fin <= rango.at[r.codigo, "max"])
    c["cobertura_clima_pct"] = pct
    c["dentro_periodo_clima"] = dentro
    c["clima_completo"] = c["cobertura_clima_pct"] >= COBERTURA_MIN
    n = c["cobertura_clima_pct"].isna().sum()
    if n:
        print(f"{n} campañas con la siembra posterior al último reporte: sin cobertura calculada")
    return c


todas = cobertura(campanias(fen), clima)
camp = todas[todas["cultivo_norm"].str.contains(CULTIVO, regex=True)] if CULTIVO else todas
print(f"{len(todas)} campañas (todos los cultivos) | {len(camp)} del cultivo seleccionado")
camp.head()

Control: para el maíz, el total debe coincidir (o casi) con `fases_maiz.xlsx` en las mismas estaciones.
La diferencia posible viene de que aquí `maiz`, `maiz amarillo`, etc. son cultivos distintos de una misma
estación, y de los registros sin fecha de siembra.

In [ ]:
fm = pd.read_excel(PROC / "fases_maiz.xlsx", sheet_name="campanias")
fm = fm.merge(est_clima, left_on=["estacion", "ddrr"], right_on=["estacion_fenologia", "ddrr"])
n_maiz = todas["cultivo_norm"].str.contains(r"\bmaiz\b|choclo").sum()
print(f"fases_maiz.xlsx: {len(fm)} campañas de maíz en estaciones con clima | este notebook: {n_maiz}")

## 3. Resumen por estación

In [ ]:
def resumen_estacion(c):
    g = c.groupby(["ddrr", "estacion_norm"])
    r = g.agg(
        codigo_clima=("codigo", "first"),
        campanias=("fecha_siembra", "size"),
        campanias_clima_completo=("clima_completo", "sum"),
        variedades=("variedad", "nunique"),
        lista_variedades=("variedad", lambda s: ", ".join(f"{v} ({n})" for v, n in s.value_counts().items())),
        anios_siembra=("fecha_siembra", lambda s: ", ".join(map(str, sorted(s.dt.year.unique())))),
        cobertura_media_pct=("cobertura_clima_pct", "mean"),
    ).reset_index()
    r["cobertura_media_pct"] = r["cobertura_media_pct"].round(1)
    # Variedades con 2 o más campañas con clima completo: útiles para comparar entre años
    rep = (c[c["clima_completo"]].groupby(["ddrr", "estacion_norm", "variedad"]).size()
             .loc[lambda s: s >= 2].groupby(level=[0, 1]).size().rename("variedades_con_2+_campanias_clima"))
    r = r.merge(rep.reset_index(), on=["ddrr", "estacion_norm"], how="left")
    r["variedades_con_2+_campanias_clima"] = r["variedades_con_2+_campanias_clima"].fillna(0).astype(int)
    return r.rename(columns={"estacion_norm": "estacion"}).sort_values(
        ["campanias", "variedades"], ascending=False).reset_index(drop=True)


por_estacion = resumen_estacion(camp)
# Estaciones con clima pero sin campañas del cultivo
sin_camp = est_clima[~est_clima.set_index(["estacion_fenologia", "ddrr"]).index.isin(
    por_estacion.set_index(["estacion", "ddrr"]).index)]
print(f"{len(por_estacion)} estaciones con campañas | {por_estacion['campanias'].sum()} campañas | "
      f"{por_estacion['campanias_clima_completo'].sum()} con clima ≥ {COBERTURA_MIN} %")
print("Estaciones con clima sin campañas del cultivo:", ", ".join(sin_camp["estacion_fenologia"]) or "ninguna")
por_estacion

In [ ]:
# Campañas por estación: total y con clima completo
d = por_estacion.sort_values("campanias")
fig, ax = plt.subplots(figsize=(8, 0.28 * len(d) + 1.2))
ax.barh(d["estacion"], d["campanias"], color="#c9c8c2", height=0.7, label="campañas")
ax.barh(d["estacion"], d["campanias_clima_completo"], color="#2a78d6", height=0.7,
        label=f"con clima ≥ {COBERTURA_MIN} %")
for y, (t, v) in enumerate(zip(d["campanias"], d["variedades"])):
    ax.text(t + 0.15, y, f"{t}  ·  {v} var.", va="center", fontsize=8, color="#52514e")
ax.set_xlabel("campañas", color="#52514e")
ax.set_title("Campañas por estación (total y con clima completo) y número de variedades",
             loc="left", fontsize=11, color="#0b0b0b")
ax.tick_params(labelsize=8, colors="#52514e")
for s in ["top", "right", "left"]:
    ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#d5d4cf")
ax.grid(axis="x", color="#ecebe7", linewidth=0.8)
ax.set_axisbelow(True)
ax.legend(loc="lower right", frameon=False, fontsize=8)
ax.set_xlim(0, d["campanias"].max() + 3)
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

## 4. Detalle por estación y variedad

In [ ]:
por_variedad = (camp.groupby(["ddrr", "estacion_norm", "variedad"], dropna=False)
                .agg(campanias=("fecha_siembra", "size"),
                     campanias_clima_completo=("clima_completo", "sum"),
                     fechas_siembra=("fecha_siembra",
                                     lambda s: ", ".join(f.strftime("%d/%m/%Y") for f in sorted(s))),
                     cobertura_pct=("cobertura_clima_pct",
                                    lambda s: ", ".join(f"{x:.0f}" for x in s)))
                .reset_index().rename(columns={"estacion_norm": "estacion"})
                .sort_values(["ddrr", "estacion", "campanias"], ascending=[True, True, False]))
por_variedad

## 5. Todos los cultivos

Mismo conteo para todos los cultivos de las estaciones con clima. En frutales y cultivos perennes la
"campaña" es la plantación (fecha de siembra antigua), así que se cuentan como una sola.

In [ ]:
cultivos = (todas.groupby(["ddrr", "estacion_norm", "cultivo_norm"])
            .agg(campanias=("fecha_siembra", "size"),
                 variedades=("variedad", "nunique"),
                 perenne_o_antigua=("perenne_o_antigua", "any"),
                 campanias_clima_completo=("clima_completo", "sum"))
            .reset_index().rename(columns={"estacion_norm": "estacion", "cultivo_norm": "cultivo"}))
resumen_cultivos = (cultivos.groupby(["ddrr", "estacion"])
                    .agg(cultivos=("cultivo", "nunique"),
                         campanias=("campanias", "sum"),
                         variedades=("variedades", "sum"),
                         lista_cultivos=("cultivo", lambda s: ", ".join(s)))
                    .reset_index().sort_values("campanias", ascending=False))
resumen_cultivos

## 6. Guardar

In [ ]:
salida = PROC / "campanias_estaciones_clima.xlsx"
with pd.ExcelWriter(salida) as xw:
    por_estacion.to_excel(xw, sheet_name="por_estacion", index=False)
    por_variedad.to_excel(xw, sheet_name="por_estacion_variedad", index=False)
    (camp.drop(columns=["inicio", "fin"]).rename(columns={"estacion_norm": "estacion", "cultivo_norm": "cultivo"})
         .to_excel(xw, sheet_name="campanias", index=False))
    resumen_cultivos.to_excel(xw, sheet_name="todos_cultivos", index=False)
    cultivos.to_excel(xw, sheet_name="todos_cultivos_detalle", index=False)
print("->", salida)